# 02_cuVS / 00 — Build the dataset: Wikipedia + Gemini Embedding 2

Every cuVS notebook in this folder searches the **same real embeddings**, built once here:

| | |
|---|---|
| Corpus | English Wikipedia (`wikimedia/wikipedia`, 2023-11-01 dump), split into ~500-token passages |
| Passages | 250,000 (at most 8 per article, so many topics are covered) |
| Queries | 10,000 real search questions from Natural Questions |
| Model | `gemini-embedding-2`, **3,072 dimensions** (the maximum), normalized |
| Stored as | float16 shards (≈1.5 GB for the corpus), loaded as float32 later |

**Cost.** About 125M tokens at the standard price of about $0.20 per 1M tokens is **about $25**. Section 5 measures real token counts on a sample and prints the estimate. Nothing is sent until you set `I_ACCEPT_COST = True`. Check current prices at ai.google.dev before running.

**Restartable.** Embeddings are saved in shards of 5,000 passages. If the runtime disconnects, rerun the notebook and it continues from the last finished shard, without paying twice.

**API key.** The notebook reads `GEMINI_API_KEY` from Colab Secrets or an environment variable, or asks for it with a hidden prompt. Never paste the key into a cell.

## 1. Install and configure

In [ ]:
import importlib.util, subprocess, sys

def has(module):
    try:
        return importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        return False

missing = [pkg for pkg, mod in {"google-genai": "google.genai", "datasets": "datasets"}.items() if not has(mod)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("installed:", missing or "nothing needed")

In [ ]:
import os, json, time, math, glob
import numpy as np, pandas as pd

MODEL        = "gemini-embedding-2"
DIM          = 3072            # maximum output dimensionality
N_PASSAGES   = 250_000
CHUNK_WORDS  = 375             # ~500 tokens (1 token is about 0.75 English words)
MIN_WORDS    = 100             # drop tiny fragments
MAX_PER_ARTICLE = 8            # spread passages over many articles
N_QUERIES    = 10_000
SHARD_SIZE   = 5_000           # passages per saved shard
BATCH_SIZE   = 50              # texts per API request
WORKERS      = 4               # parallel requests
PRICE_PER_M_TOKENS = 0.20      # USD, standard (non-batch) price; check ai.google.dev/pricing

I_ACCEPT_COST = False          # set to True after reading the estimate in section 5

## 2. Where to store the data

Colab's local disk is wiped when the runtime is recycled. The notebook tries to mount **Google Drive** so the embeddings survive. If that isn't possible (for example from some VS Code kernels), it falls back to local disk and warns you. Section 8 has an optional backup to a private Hugging Face dataset.

In [ ]:
STORE_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE_DIR = "/content/drive/MyDrive/cuda-rapids/cuvs_wiki_gemini3072"
except Exception as e:
    print(f"Google Drive not available ({type(e).__name__}: {e})")

if STORE_DIR is None:
    STORE_DIR = os.path.abspath("cuvs_wiki_gemini3072")
    print("WARNING: using local disk. Files are lost when the Colab runtime is recycled.")

EMB_DIR = os.path.join(STORE_DIR, "corpus_shards")
os.makedirs(EMB_DIR, exist_ok=True)
print("STORE_DIR =", STORE_DIR)

## 3. API key and a first call

In [ ]:
def get_api_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass
    if os.environ.get("GEMINI_API_KEY"):
        return os.environ["GEMINI_API_KEY"]
    from getpass import getpass
    return getpass("GEMINI_API_KEY: ")

from google import genai
from google.genai import types

client = genai.Client(api_key=get_api_key())

`gemini-embedding-2` has no `task_type` setting. Instead, the text itself carries a task prefix:
- documents: `title: {title} | text: {content}`
- queries: `task: search result | query: {content}`

Each input must be wrapped in its own `types.Content`. Otherwise a list of strings is merged into **one** embedding.

In [ ]:
def doc_text(title, text):
    return f"title: {title} | text: {text}"

def query_text(q):
    return f"task: search result | query: {q}"

def embed(texts, retries=8):
    """Embed a list of strings -> float32 array (len(texts), DIM). Retries on rate limits/errors."""
    contents = [types.Content(parts=[types.Part(text=t)]) for t in texts]
    for attempt in range(retries):
        try:
            resp = client.models.embed_content(
                model=MODEL, contents=contents,
                config=types.EmbedContentConfig(output_dimensionality=DIM))
            out = np.asarray([e.values for e in resp.embeddings], dtype=np.float32)
            assert out.shape == (len(texts), DIM), out.shape
            return out
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait = min(60, 2 ** attempt)
            print(f"  retry {attempt + 1} in {wait}s ({type(e).__name__}: {str(e)[:120]})")
            time.sleep(wait)

test = embed([doc_text("Tesla T4", "The T4 is a data-center GPU based on the Turing architecture."),
              doc_text("Mango", "The mango is an edible stone fruit."),
              query_text("which nvidia gpu uses turing")])
print("shape:", test.shape, "| norms:", np.linalg.norm(test, axis=1).round(4))
print("query·GPU doc  :", round(float(test[2] @ test[0]), 4))
print("query·mango doc:", round(float(test[2] @ test[1]), 4))

## 4. Build the passage corpus

Wikipedia is streamed, so the full dump is never downloaded. Each article is split on paragraph boundaries into chunks of about 375 words (≈500 tokens).

In [ ]:
from datasets import load_dataset

passages_path = os.path.join(STORE_DIR, "passages.parquet")

def chunk_article(text):
    chunks, cur = [], []
    for para in text.split("\n"):
        words = para.split()
        if not words:
            continue
        cur.extend(words)
        while len(cur) >= CHUNK_WORDS:
            chunks.append(" ".join(cur[:CHUNK_WORDS]))
            cur = cur[CHUNK_WORDS:]
    if len(cur) >= MIN_WORDS:
        chunks.append(" ".join(cur))
    return chunks

if os.path.exists(passages_path):
    passages = pd.read_parquet(passages_path)
    print(f"loaded {len(passages):,} passages")
else:
    wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
    rows, n_articles, t0 = [], 0, time.time()
    for art in wiki:
        n_articles += 1
        for j, ch in enumerate(chunk_article(art["text"])[:MAX_PER_ARTICLE]):
            rows.append((art["title"], j, ch, len(ch.split())))
        if len(rows) >= N_PASSAGES:
            break
        if n_articles % 20_000 == 0:
            print(f"  {n_articles:,} articles -> {len(rows):,} passages ({time.time() - t0:.0f}s)")
    passages = pd.DataFrame(rows[:N_PASSAGES], columns=["title", "chunk", "text", "n_words"])
    passages.insert(0, "pid", np.arange(len(passages), dtype=np.int32))
    passages.to_parquet(passages_path)
    print(f"{n_articles:,} articles -> {len(passages):,} passages, saved")

print(f"{passages['title'].nunique():,} articles | words per passage: "
      f"median {passages['n_words'].median():.0f}, mean {passages['n_words'].mean():.0f}")
passages.head(3)

## 5. Cost estimate

We count real tokens on a random sample of 200 passages with the API's token counter, then extrapolate. If token counting isn't available for this model, the notebook falls back to words ÷ 0.75.

In [ ]:
sample = passages.sample(200, random_state=0)
try:
    sample_tokens = sum(
        client.models.count_tokens(model=MODEL, contents=doc_text(r.title, r.text)).total_tokens
        for r in sample.itertuples())
    how = "measured with count_tokens"
except Exception as e:
    sample_tokens = int(sum(sample["n_words"]) / 0.75)
    how = f"estimated from words ({type(e).__name__})"

tokens_per_passage = sample_tokens / len(sample)
total_tokens = tokens_per_passage * len(passages) + N_QUERIES * 20   # queries are ~20 tokens each
print(f"tokens per passage : {tokens_per_passage:.0f}  ({how})")
print(f"total tokens       : {total_tokens / 1e6:.1f} M")
print(f"estimated cost     : ${total_tokens / 1e6 * PRICE_PER_M_TOKENS:.2f}  at ${PRICE_PER_M_TOKENS}/M tokens")
print(f"vectors            : {len(passages):,} x {DIM} -> {len(passages) * DIM * 4 / 1e9:.2f} GB float32, "
      f"{len(passages) * DIM * 2 / 1e9:.2f} GB float16 on disk")

## 6. Embed the corpus

Shards that already exist are skipped, so this cell is safe to rerun after a disconnect.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

assert I_ACCEPT_COST, "Read the estimate above, then set I_ACCEPT_COST = True in section 1 and rerun."

n_shards = math.ceil(len(passages) / SHARD_SIZE)
t_start, done_now = time.time(), 0
with ThreadPoolExecutor(WORKERS) as pool:
    for s in range(n_shards):
        path = os.path.join(EMB_DIR, f"shard_{s:05d}.npy")
        if os.path.exists(path):
            continue
        part = passages.iloc[s * SHARD_SIZE:(s + 1) * SHARD_SIZE]
        texts = [doc_text(t, x) for t, x in zip(part["title"], part["text"])]
        batches = [texts[i:i + BATCH_SIZE] for i in range(0, len(texts), BATCH_SIZE)]
        vecs = np.concatenate(list(pool.map(embed, batches)))
        np.save(path + ".tmp.npy", vecs.astype(np.float16))
        os.replace(path + ".tmp.npy", path)          # only complete shards get the final name
        done_now += len(part)
        rate = done_now / (time.time() - t_start)
        left = (len(passages) - (s + 1) * SHARD_SIZE) / max(rate, 1e-9)
        print(f"shard {s + 1:>3}/{n_shards}  {rate:6.0f} passages/s  ~{left / 60:5.1f} min left")

print("corpus shards:", len(glob.glob(os.path.join(EMB_DIR, "shard_*.npy"))), "/", n_shards)

## 7. Queries

10,000 real search questions from Natural Questions (Google search queries answered by Wikipedia). They aren't matched to our passages one-to-one. For cuVS we measure how well approximate search matches **exact** search, so any realistic queries work.

In [ ]:
queries_path, q_emb_path = os.path.join(STORE_DIR, "queries.parquet"), os.path.join(STORE_DIR, "queries_emb.npy")

if not os.path.exists(queries_path):
    nq = load_dataset("sentence-transformers/natural-questions", split="train")
    qs = pd.Series(nq["query"]).drop_duplicates().sample(N_QUERIES, random_state=0).reset_index(drop=True)
    pd.DataFrame({"qid": np.arange(len(qs), dtype=np.int32), "query": qs}).to_parquet(queries_path)
queries = pd.read_parquet(queries_path)

if not os.path.exists(q_emb_path):
    assert I_ACCEPT_COST, "Set I_ACCEPT_COST = True first."
    texts = [query_text(q) for q in queries["query"]]
    with ThreadPoolExecutor(WORKERS) as pool:
        q = np.concatenate(list(pool.map(embed, [texts[i:i + BATCH_SIZE] for i in range(0, len(texts), BATCH_SIZE)])))
    np.save(q_emb_path, q.astype(np.float16))
print(f"{len(queries):,} queries embedded")
queries.head()

## 8. Assemble, check and back up

In [ ]:
corpus_path = os.path.join(STORE_DIR, "corpus_emb.npy")
if not os.path.exists(corpus_path):
    shards = sorted(glob.glob(os.path.join(EMB_DIR, "shard_*.npy")))
    assert len(shards) == math.ceil(len(passages) / SHARD_SIZE), "some shards are missing: rerun section 6"
    np.save(corpus_path, np.concatenate([np.load(p) for p in shards]))

X = np.load(corpus_path, mmap_mode="r")
Q = np.load(q_emb_path)
norms = np.linalg.norm(np.asarray(X[:10_000], dtype=np.float32), axis=1)
print(f"corpus  {X.shape} {X.dtype}  ({os.path.getsize(corpus_path) / 1e9:.2f} GB)")
print(f"queries {Q.shape} {Q.dtype}")
print(f"norms: min {norms.min():.4f}  max {norms.max():.4f}   (normalized -> inner product = cosine)")

meta = {"model": MODEL, "dim": DIM, "n_passages": int(X.shape[0]), "n_queries": int(Q.shape[0]),
        "chunk_words": CHUNK_WORDS, "max_per_article": MAX_PER_ARTICLE, "dtype_on_disk": "float16",
        "doc_prefix": "title: {title} | text: {content}", "query_prefix": "task: search result | query: {content}",
        "source": "wikimedia/wikipedia 20231101.en; sentence-transformers/natural-questions"}
with open(os.path.join(STORE_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
meta

In [ ]:
# Sanity check: exact search for 3 queries on the GPU
import cupy as cp

Xg = cp.asarray(np.asarray(X, dtype=np.float32))
for qi in [0, 1, 2]:
    scores = Xg @ cp.asarray(Q[qi].astype(np.float32))
    top = cp.argsort(-scores)[:3].get()
    print("Q:", queries.loc[qi, "query"])
    for r, pid in enumerate(top, 1):
        print(f"   {r}. [{float(scores[pid]):.3f}] {passages.loc[pid, 'title']}: {passages.loc[pid, 'text'][:90]}...")
del Xg

In [ ]:
# Optional: back up to a private Hugging Face dataset (needs HF_TOKEN with write access)
BACKUP_TO_HF = False
HF_REPO = "your-username/cuvs-wiki-gemini3072"    # change me

if BACKUP_TO_HF:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=STORE_DIR, repo_id=HF_REPO, repo_type="dataset",
                      ignore_patterns=["corpus_shards/*"])
    print("uploaded to", HF_REPO)

## Summary

- **Corpus:** 250k Wikipedia passages of about 500 tokens each, embedded with `gemini-embedding-2` at **3,072 dimensions**.
- **Queries:** 10k real questions from Natural Questions, embedded with the query prefix.
- **Files** in `STORE_DIR`: `passages.parquet`, `queries.parquet`, `corpus_emb.npy`, `queries_emb.npy` and `meta.json`.
- Vectors are normalized, so **inner product = cosine similarity**.

**Next:** `01_brute_force_and_ground_truth.ipynb` — exact search on the GPU and the ground truth that every index will be measured against.